In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import seaborn as sns
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder,OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')



In [ ]:




# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
path = os.path.join(path,'Q3_data.csv')
df = pd.read_csv(path)


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()
df.shape

In [ ]:
# Task 1: Write your code here:
df = df.fillna(df.median())
df.shape

In [ ]:
# Task 2: Write your code here:
# Task 3: Write your code here:
print("Checking for duplicate rows...")
duplicate_rows = df.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicate rows. Removing them...")
    df.drop_duplicates(inplace=True,)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")

df.info()

In [ ]:
# Task 3: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])

df.head()

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler
num_cols = df.select_dtypes(include=["float",'int']).columns.drop('Target')
scaler = StandardScaler()

df[num_cols] = scaler.fit_transform(df[num_cols])

df


In [ ]:
# Task 5: Write your code here:
# Checking for inbalance
def check_target_imbalance(df, target_column):
    print("Target Distribution:")
    print(df[target_column].value_counts(normalize=True))
    sns.countplot(x=df[target_column])
    plt.title("Target Distribution")
    plt.show()

check_target_imbalance(df, "Target")

# So we have some inbalance so we are going to use StratifiedKFold later

In [ ]:
# Task 1: Write your code here:
X = df.drop("Target",axis=1)
y = df['Target']

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)


# F1 will be our metric
scores_f1 = []
accuracy=[]
# StratifiedKFold because we an inbalanced dataset and not shuffling as requested
skf = StratifiedKFold(n_splits=5, shuffle=False)

for train_index, test_index in skf.split(X, y):
    X_Train, X_Test = X.loc[train_index, :], X.loc[test_index, :]
    y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]

    model.fit(X_Train, y_Train)
    y_pred = model.predict(X_Test)

    scores_f1.append(f1_score(y_Test, y_pred, average='weighted'))
    accuracy.append(accuracy_score(y_Test, y_pred))

print(f"F1-Score: {np.mean(scores_f1):.4f} accuracy_score {np.mean(accuracy):.4f}"  )

In [ ]:
# Task 1: Write your code here:

ridge_importance = list(zip(X.columns, model.feature_importances_))
sorted_ridge_importance = sorted(ridge_importance, key=lambda x: abs(x[1]), reverse=True)

features, coefficients = zip(*sorted_ridge_importance)

plt.figure(figsize=(10, 6))
plt.barh(features, coefficients, color='darkblue')
plt.xlabel('Coefficient Value')
plt.ylabel('Features')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task Bonus: Write your code here: